# 02 — λ: the price of admission

**2026 Quantum Accelerator · Day 1 · 11:00 · 30 minutes**

In the last notebook we set one number generously and moved on:

```python
LAMBDA = 2.0 * float(values.sum())      # generous on purpose
```

That number is how much the solver is charged for breaking the capacity rule. It is not a
detail. It is the price you pay for the fact that **a QUBO cannot represent a constraint —
only a cost for breaking one.**

This module is thirty minutes on that one number, because it is the single most
transferable skill in this field, and because it decides whether everything else today
works.

**What you will type.** One list of numbers.

---
## 1. The same problem, slightly bigger

Five items was right for seeing the matrix. It is too small to see λ misbehave, because
with five items almost any λ works. So we scale up — still trivial for the exact dynamic
program, which gives us a *known right answer* to measure against.

In [ ]:
import numpy as np
import time
from pyqubo import Array, Constraint, Placeholder
from dwave.samplers import SimulatedAnnealingSampler
import helpers_knapsack as hk

rng = np.random.default_rng(7)
N_ITEMS = 40
values = rng.integers(10, 100, N_ITEMS)
weights = rng.integers(5, 50, N_ITEMS)
CAPACITY = int(0.5 * weights.sum())


def knapsack_dp(values, weights, capacity):
    best = np.zeros(int(capacity) + 1, dtype=np.int64)
    for v, w in zip(values, weights):
        w = int(w)
        if w > capacity:
            continue
        best[w:] = np.maximum(best[w:], best[:-w] + int(v))
    return int(best[int(capacity)])


OPTIMUM = knapsack_dp(values, weights, CAPACITY)
print(f"{N_ITEMS} items, capacity {CAPACITY} of {int(weights.sum())} total weight")
print(f"exact optimum (dynamic programming): {OPTIMUM}")
print(f"largest single value: {int(values.max())}   total value: {int(values.sum())}")

---
## 2. Build it once, turn the dial afterwards

The important mechanical trick: wrap the rule in a **named** `Constraint`, and make its
weight a **`Placeholder`**. Then `compile()` runs once, and every λ you try afterwards is
just a different `feed_dict` — no rebuilding, no waiting.

The name matters as much as the placeholder. It is what lets the solver tell us *which*
rule it broke, instead of only returning a bad number.

In [ ]:
def slack_bits(capacity):
    coeffs, remaining, power = [], int(capacity), 1
    while remaining > 0:
        c = min(power, remaining)
        coeffs.append(c)
        remaining -= c
        power *= 2
    return coeffs


COEFFS = slack_bits(CAPACITY)
x = Array.create("x", shape=N_ITEMS, vartype="BINARY")
s = Array.create("s", shape=len(COEFFS), vartype="BINARY")

objective = -sum(int(v) * xi for v, xi in zip(values, x))
load = sum(int(w) * xi for w, xi in zip(weights, x))
slack = sum(int(c) * si for c, si in zip(COEFFS, s))

MODEL = (objective
         + Placeholder("lam") * Constraint((load + slack - int(CAPACITY)) ** 2,
                                           label="capacity")).compile()

print(f"compiled once: {N_ITEMS} item variables + {len(COEFFS)} slack bits")
print("lambda is now a dial, not a rebuild")

In [ ]:
SAMPLER = SimulatedAnnealingSampler()


def solve(lam, num_reads=100, num_sweeps=2000, seed=1):
    """Anneal at one value of lambda. Returns feasibility and the best legal value."""
    feed = {"lam": float(lam)}
    t0 = time.perf_counter()
    ss = SAMPLER.sample(MODEL.to_bqm(feed_dict=feed), num_reads=num_reads,
                        num_sweeps=num_sweeps, seed=seed)
    decoded = MODEL.decode_sampleset(ss, feed_dict=feed)
    wall = time.perf_counter() - t0

    legal = [d for d in decoded if not d.constraints(only_broken=True)]
    if legal:
        vals = [sum(int(v) * d.array("x", i) for i, v in enumerate(values)) for d in legal]
        k = int(np.argmax(vals))
        best_value = vals[k]
        best_weight = sum(int(w) * legal[k].array("x", i) for i, w in enumerate(weights))
    else:
        best_value = best_weight = None

    return dict(lam=lam, wall=wall,
                feasible_pct=100 * len(legal) / len(decoded),
                best_value=best_value, best_weight=best_weight,
                example_broken=(decoded[0].constraints(only_broken=True)
                                if not legal else {}))

---
## 3. Two λ values, by hand

Before sweeping, look at the two failure modes one at a time. Run this cell as it is.

In [ ]:
for lam in (0.01, 100.0):
    r = solve(lam)
    print(f"lambda = {lam}")
    print(f"  legal runs : {r['feasible_pct']:.0f}%")
    if r["best_value"] is None:
        print(f"  best legal : none")
        print(f"  what broke : {r['example_broken']}")
    else:
        print(f"  best legal : {r['best_value']}  (weight {r['best_weight']}/{CAPACITY})")
        print(f"  gap to the exact optimum {OPTIMUM}: "
              f"{100 * (OPTIMUM - r['best_value']) / OPTIMUM:.1f}%")
    print()

Notice what the low-λ case printed: not a mysterious bad number, but the **name of the
rule that was broken**. `decode_sampleset` gives you that for free, for every named
constraint, on every run. When you have six rules instead of one — which you will, after
lunch — this is the difference between debugging and guessing.

---
## 4. Exercise — sweep it

**You type one list.** Ten values spanning several orders of magnitude. A few seconds.

> **Calibrated 6 October 2026** against this exact instance with `calibrate_lambda.py`.
> The grid below straddles both arms. Expect roughly: nothing legal below λ=0.1; the exact
> optimum found at λ=0.1 but in only ~9% of runs; ~78% legal by λ=20, at a ~21% gap.
> The whole sweep runs in about 3 seconds.

In [ ]:
LAMBDAS = [0.01, 0.03, 0.1, 0.2, 0.3, 0.5, 1, 2, 5, 20, 100]     # <-- CHANGE ME

feas, gaps = [], []
for lam in LAMBDAS:
    r = solve(lam)
    feas.append(r["feasible_pct"])
    gap = None if r["best_value"] is None else 100 * (OPTIMUM - r["best_value"]) / OPTIMUM
    gaps.append(gap)
    shown = "     -" if gap is None else f"{gap:5.2f}%"
    print(f"  lambda {lam:>8g}   legal {r['feasible_pct']:>5.1f}%   "
          f"best {str(r['best_value']):>6}   gap {shown}   [{r['wall']:.2f}s]")

hk.plot_lambda_sweep(LAMBDAS, feas, gaps)

### What the two arms mean

**Left arm — λ too small.** Breaking the capacity rule is cheaper than leaving value
behind, so the solver overfills the bag. It is not malfunctioning; it is optimising the
objective you actually wrote down.

**Right arm — λ too large.** Every run is legal, and the answers get worse. The penalty
term dominates the energy landscape, and the differences between a good packing and a
mediocre one become numerical noise by comparison. The solver spends its effort satisfying
a rule it has already satisfied.

**And the part that is more interesting than a "sweet spot".** Look at the two curves
together. Feasibility climbs steadily with λ. Quality gets steadily *worse*. They do not
peak in the same place — so there is no single best λ, there is a **trade-off between
reliability and quality**, and which end you want depends on your budget.

At λ=0.1 this instance gives you the *exact optimum* — in about one run in eleven. At λ=20
you get a legal roster four times out of five, and it is a fifth worse than optimal. Low λ
is a lottery ticket; high λ is an insurance policy.

The mechanism is worth a sentence, because it transfers. At low λ the solver cheerfully
overfills the bag, so most runs are illegal — but on the rare run that lands inside the
capacity, it lands there *packed tight*, because it was never pushed away from the
boundary. At high λ the solver is shoved hard away from the limit and settles for
comfortably under-filled, safe, mediocre answers.

### And a rule of thumb you can leave with

For a knapsack this can be reasoned about rather than guessed: if λ exceeds the largest
possible gain from cheating, cheating can never pay. Taking one extra item gains at most
`max(values)`, so any λ above that is *provably* sufficient to make the optimum feasible.

```
lambda > max(values)      # sufficient for knapsack
```

Computable bounds of this kind exist more generally — Glover, Kochenberger & Du's QUBO
tutorial (arXiv:1811.11538) is the standard reference, and there is a body of work on
tighter penalty bounds. You do not have to guess. But note what the sweep just showed:
*provably sufficient* is not the same as *good for the solver*. The bound tells you where
the left arm ends. It says nothing about the right arm.

There is also a way to avoid slack variables entirely for inequalities, called
**unbalanced penalisation** (arXiv:2211.13914), which trades one coefficient for two and
drops the extra bits. Worth knowing it exists; not worth your next twenty minutes.

---
## 5. Why this is the most important half hour of the day

You have one rule and one dial. This afternoon you will have **six rules and six dials**,
and you will find that:

- feasibility collapses as rules accumulate, even with the variable count fixed;
- re-balancing *one* dial recovers more than a sixteen-fold increase in compute does;
- and a constraint solver that never had to flatten a rule into a cost solves the whole
  thing in milliseconds.

That last point is the honest frame for the day. λ exists because we gave up the ability
to state a constraint. Everything good and everything bad about the QUBO form follows from
that one trade.

Which raises the obvious question, and we will answer it at 16:00: **do you always have to
tune this?** No. Some solvers take linear constraints directly, and you will see the
difference.

---
## 6. Stretch goals

**A. Per-rule dials.** Add the exclusivity rule from notebook 01 stretch D
(`Constraint(x[0] * x[1], label="either_or")`) with its *own* `Placeholder`. Sweep the two
weights independently. Does the best pair lie where you would have guessed from sweeping
each alone?

**B. Does effort substitute for tuning?** Take the worst λ from your sweep that still
produced a legal answer, and increase `num_sweeps` by 10x and 100x. How much of the gap
can you buy back with compute instead of thought? (You will see this question again at
13:00, with a sharper answer.)

**C. Scale and re-sweep.** Set `N_ITEMS = 200`. Does the best λ move? Does the *shape* of
the curve change? Predict before you run.

**D. Drop the slack.** Read the unbalanced-penalisation paper's formulation
(arXiv:2211.13914) and implement it for this instance: no slack bits, two coefficients.
Compare variable count and best value against what you have.